# Tier 3: Molecular Modeling & Force Fields
## Lab 2.3 — Molecular Docking, Semi-Empirical Scoring Functions, and Virtual Screening

Welcome to **Lab 2.3** of the *Computational Structural Biology and Generative Macromolecular Design* curriculum!

### Why This Lab Matters
In Labs 2.0 through 2.2, we explored all-atom force fields, energy minimization, explicit solvation, and statistical mechanics trajectories. Now, we turn to one of the most widely applied computational disciplines in pharmacology and bioengineering: **Molecular Docking and Virtual Screening**.

Molecular docking predicts the preferred 3D pose, conformation, and free energy of binding ($\Delta G_{	ext{bind}}$) of small-molecule ligands interacting with macromolecular binding pockets:
1. **Conformational Search Mechanics:** The conformational degrees of freedom of a flexible small molecule grow exponentially with the number of rotatable dihedral bonds ($N_{	ext{conf}} \sim 3^{N_{	ext{rot}}}$). Exhaustive systematic grid search is physically impossible; docking algorithms rely on stochastic global optimization heuristics (e.g., genetic algorithms, Lamarckian genetic algorithms, Monte Carlo simulated annealing).
2. **Semi-Empirical Scoring Functions:** Physics-based force fields evaluate full non-bonded electrostatics and van der Waals energies, but lack terms for the entropic loss of freezing rotatable bonds ($\Delta S_{	ext{rot}}$) and desolvation penalties. Semi-empirical scoring functions (such as AutoDock Vina) train empirical weights against experimental dissociation constants ($K_d, K_i$):
   $$\Delta G_{	ext{bind}} pprox \sum w_i f_i(\mathbf{r})$$
3. **Rigid Receptor vs. Induced Fit:** Standard high-throughput docking treats the protein receptor as a rigid coordinate grid, rotating only ligand torsions. While fast, this neglects active-site side-chain remodeling (induced fit).
4. **Ligand Chemical Curation & Tautomerism:** Docking requires rigorous chemical preparation (RDKit): adding missing hydrogens at physiological pH 7.4, generating plausible 3D conformers (ETKDGv3), computing Gasteiger or AM1-BCC partial charges, and verifying correct protonation states.
5. **Evaluating Virtual Screening Ensembles:** A screening campaign ranks thousands of candidate molecules against a receptor target. We quantify screening efficacy using **Receiver Operating Characteristic (ROC)** curves, **Area Under the Curve (AUC)**, and early-enrichment metrics (**Enrichment Factors**, $	ext{EF}_{1\%}, 	ext{EF}_{5\%}$).

---

### Learning Objectives
By the end of this laboratory, you will be able to:
- Formulate the thermodynamic components of semi-empirical docking scoring functions (steric repulsion, hydrophobic contact, hydrogen bonding, and torsional entropy loss).
- Curate small-molecule ligands using RDKit: parse SMILES, assign 3D coordinates via ETKDG, calculate rotatable bond counts, and compute molecular weights and $\log P$.
- Define active-site search bounding boxes ($X, Y, Z$ center and dimensional extents in Ångströms) centered on catalytic residues in *Ideonella sakaiensis* PETase (PDB: `6ANE`).
- Implement an automated Python docking scoring engine evaluating ligand pose interactions against rigid protein grids.
- Generate synthetic decoy and active ligand libraries, compute binding scores, and calculate classification performance metrics (ROC-AUC, $	ext{EF}_{1\%}$, and $	ext{EF}_{5\%}$).
- Visualize 3D docked poses, polar hydrogen bonds, and steric fit inside active sites using `py3Dmol`.
- Self-assess your mastery using interactive `jupyterquiz` questions.

---

### Running Systems in Tier 3
- **Protein System:** *Ideonella sakaiensis* **PETase** (PDB: `6ANE`), docking model polyethylene terephthalate fragments (MHET, BHET, and structural substrate analogs) against the catalytic triad ($	ext{Ser160}$–$	ext{Asp206}$–$	ext{His237}$) and aromatic clamp ($	ext{Trp185}$).
- **Nucleic Acid System:** *Vibrio vulnificus* *add* **A-riboswitch** (PDB: `1Y26`), exploring purine base analog discrimination (adenine vs. guanine, hypoxanthine) in an RNA aptamer pocket.

Let's begin by preparing our Google Colab computational environment.


### Section 0: Environment Setup and Pinned Dependencies
We pin all core packages, installing RDKit (the cheminformatics standard for molecular sanitization, conformer generation, and fingerprinting) alongside our structural biology and visualization libraries.


In [ ]:
# Install required scientific bioinformatics and cheminformatics packages
# Use rdkit>=2024.3 and numpy>=2.1.0 for Python 3.13 / NumPy 2.x C-ABI compatibility
!pip install -q rdkit>=2024.3 "numpy>=2.1.0" "pandas>=2.2.0" "biopython>=1.83" jupyterquiz==2.8.1 py3Dmol==2.1.0 scikit-learn scipy matplotlib requests

import os
import sys
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import py3Dmol

# RDKit imports for cheminformatics
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, rdMolTransforms
from rdkit.Chem.rdchem import Mol

# BioPython and sklearn imports
from Bio.PDB import PDBParser
from sklearn.metrics import roc_curve, auc
from IPython.display import display

# Pinned course reproducibility seed
COURSE_SEED = 42
random.seed(COURSE_SEED)
np.random.seed(COURSE_SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("manifests", exist_ok=True)

print("Lab 2.3 environment initialized successfully.")
print(f"Python version: {sys.version.split()[0]}")
print(f"RDKit version: {rdkit.__version__}")
print(f"NumPy version: {np.__version__} | Pandas version: {pd.__version__}")
print(f"Reproducibility seed: {COURSE_SEED}")


---
## Section 1: The Physics and Thermodynamics of Docking Scoring Functions

Molecular docking seeks to predict both the correct binding pose and the **free energy of binding** ($\Delta G_{	ext{bind}}$):
$$\Delta G_{	ext{bind}} = -R T \ln(K_a) = R T \ln(K_d)$$

### 1. The Anatomy of an Empirical Scoring Function (Vina Model)
In software like AutoDock Vina, the total binding score is calculated as a sum over all intermolecular atom pairs ($i \in 	ext{ligand}, j \in 	ext{receptor}$):
$$\Delta G_{	ext{bind}} pprox \sum_{i < j} f_{	ext{type}(i), 	ext{type}(j)}(r_{ij}) + w_{	ext{rot}} N_{	ext{rot}}$$

The pairwise interaction function $f(r)$ decomposes into five fundamental chemical potentials:
1. **Gaussian Attractive Terms ($	ext{gauss}_1, 	ext{gauss}_2$):** Empirical dispersion and attractive dipole interactions:
   $$f_{	ext{gauss}}(r) = w_{	ext{gauss}} e^{-(r / c)^2}$$
2. **Repulsive Steric Wall ($	ext{repulsion}$):** Penalizes atomic steric overlap when distance $r < r_{	ext{vdw}}(i) + r_{	ext{vdw}}(j)$:
   $$f_{	ext{repulsion}}(r) = w_{	ext{rep}} (d)^2 \quad 	ext{if } d < 0, \quad 	ext{where } d = r - (R_i + R_j)$$
3. **Hydrophobic Contact ($	ext{hydrophobic}$):** Favorable piecewise linear term rewarding contacts between apolar carbon and halogen atoms:
   $$f_{	ext{hydrophobic}}(r) = w_{	ext{hydro}} 	imes h(d)$$
4. **Hydrogen Bonding ($	ext{hbond}$):** Piecewise linear term rewarding ideal donor-acceptor distances ($r pprox 2.8 - 3.2\,	ext{Å}$) between polar heteroatoms ($	ext{O}, 	ext{N}$):
   $$f_{	ext{hbond}}(r) = w_{	ext{hbond}} 	imes b(d)$$
5. **Torsional Entropy Penalty ($N_{	ext{rot}}$):**  
   Freezing a flexible rotatable single bond upon binding to a rigid receptor incurs an entropic free energy cost:
   $$\Delta G_{	ext{rot}} = w_{	ext{rot}} N_{	ext{rot}}$$
   where $N_{	ext{rot}}$ is the number of unhindered, non-terminal rotatable dihedral bonds (typically $w_{	ext{rot}} pprox +0.05 - 0.10\,	ext{kcal}/(	ext{mol}\cdot	ext{rotator})$).

Let's implement a clean, pure-Python AutoDock Vina-style pairwise scoring function.


In [ ]:
# Pure-Python implementation of AutoDock Vina pairwise empirical potentials
def vina_distance_d(r: float, r_vdw1: float, r_vdw2: float) -> float:
    """Surface distance: d = r - (R1 + R2)"""
    return r - (r_vdw1 + r_vdw2)

def vina_gauss1(d: float) -> float:
    return math.exp(-(d / 0.5)**2)

def vina_gauss2(d: float) -> float:
    return math.exp(-((d - 3.0) / 2.0)**2)

def vina_repulsion(d: float) -> float:
    return (d**2) if d < 0.0 else 0.0

def vina_hydrophobic(d: float) -> float:
    if d <= 0.5:
        return 1.0
    elif d >= 1.5:
        return 0.0
    else:
        return 1.5 - d

def vina_hbond(d: float) -> float:
    if d <= -0.7:
        return 1.0
    elif d >= 0.0:
        return 0.0
    else:
        return -d / 0.7

def evaluate_pairwise_vina_score(r: np.ndarray, is_hydrophobic: bool = False, 
                                 is_hbond: bool = False, r_sum_vdw: float = 3.6) -> np.ndarray:
    """
    Evaluates combined empirical Vina interaction potential over interatomic distance array r (Å).
    Typical Vina weights:
      w_gauss1 = -0.035579
      w_gauss2 = -0.005156
      w_rep    = +0.840245
      w_hydro  = -0.035069
      w_hbond  = -0.587439
    """
    w_g1 = -0.035579
    w_g2 = -0.005156
    w_rep = 0.840245
    w_hydro = -0.035069
    w_hbond = -0.587439
    
    scores = []
    for dist in r:
        d = dist - r_sum_vdw
        val = w_g1 * vina_gauss1(d) + w_g2 * vina_gauss2(d) + w_rep * vina_repulsion(d)
        if is_hydrophobic:
            val += w_hydro * vina_hydrophobic(d)
        if is_hbond:
            val += w_hbond * vina_hbond(d)
        scores.append(val)
    return np.array(scores)

# Distance sweep from 1.5 Å to 6.0 Å
r_grid = np.linspace(1.8, 6.0, 200)

score_generic = evaluate_pairwise_vina_score(r_grid, r_sum_vdw=3.6)
score_hydro = evaluate_pairwise_vina_score(r_grid, is_hydrophobic=True, r_sum_vdw=3.6)
score_hbond = evaluate_pairwise_vina_score(r_grid, is_hbond=True, r_sum_vdw=3.2)

plt.figure(figsize=(8, 4.2), dpi=100)
plt.plot(r_grid, score_generic, color='#1f77b4', lw=2.0, label='General C-C Contact')
plt.plot(r_grid, score_hydro, color='#ff7f0e', lw=2.0, label='Hydrophobic Contact')
plt.plot(r_grid, score_hbond, color='#2ca02c', lw=2.2, label='Hydrogen Bond (O...N, R_vdw=3.2 Å)')
plt.axhline(0, color='gray', linestyle='--', lw=1.0)
plt.axvline(3.6, color='gray', linestyle=':', label='Sum of VdW Radii (3.6 Å)')
plt.ylim(-0.8, 2.0)
plt.title("AutoDock Vina Empirical Pairwise Scoring Function Potentials", fontsize=11, fontweight="bold")
plt.xlabel("Interatomic Distance r (Å)", fontsize=10)
plt.ylabel("Pairwise Contribution (kcal/mol)", fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

print(f"Optimal Hydrogen-Bond Minimum Energy: {score_hbond.min():.3f} kcal/mol (Strong directional stabilization)")
print(f"Steric Repulsion at r = 2.0 Å:        {score_generic[r_grid < 2.05][-1]:.2f} kcal/mol (Severe steric penalty)")


### Formative Assessment: Scoring Functions & Rotatable Bonds
Test your comprehension of docking energy evaluation.


In [ ]:
from jupyterquiz import display_quiz

quiz_section_1 = [
    {
        "question": "Why do docking scoring functions (such as AutoDock Vina) penalize ligands with high numbers of rotatable bonds (w_rot * N_rot)?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Upon binding into a well-defined receptor pocket, the ligand loses conformational flexibility; freezing free dihedral rotations carries an unfavorable conformational entropy penalty (-T * Delta_S_rot > 0).",
                "correct": True,
                "feedback": "Correct! Freezing rotatable bonds reduces conformational microstates, imposing an entropic cost that counteracts favorable binding enthalpy."
            },
            {
                "answer": "Because molecules with rotatable bonds are insoluble in water.",
                "correct": False,
                "feedback": "Incorrect. Solubility is governed by polar surface area and logP, not rotatable bond count alone."
            },
            {
                "answer": "Because rotatable bonds always carry negative electrical charges.",
                "correct": False,
                "feedback": "Incorrect. Dihedral bonds are covalent geometric arrangements, not point charges."
            },
            {
                "answer": "To force the ligand to become an aromatic planar ring.",
                "correct": False,
                "feedback": "Incorrect. Rotatable bonds are a natural property of flexible acyclic chains."
            }
        ]
    },
    {
        "question": "What is the primary operational trade-off of rigid-receptor docking compared to flexible induced-fit docking?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Rigid-receptor docking precomputes static potential grids, allowing screening of thousands of molecules per hour, but fails to predict binding if the protein must remodel its side chains to accommodate the ligand.",
                "correct": True,
                "feedback": "Correct! Rigid docking provides rapid virtual screening throughput at the expense of ignoring induced-fit structural rearrangements."
            },
            {
                "answer": "Rigid-receptor docking is 100% accurate for all known proteins.",
                "correct": False,
                "feedback": "Incorrect. Rigid docking has high false-negative rates for targets with dynamic binding pockets."
            },
            {
                "answer": "Rigid-receptor docking requires quantum mechanical DFT calculations for every atom.",
                "correct": False,
                "feedback": "Incorrect. Docking uses classical semi-empirical potentials, not full quantum mechanics."
            },
            {
                "answer": "Rigid-receptor docking cannot evaluate molecules with molecular weight greater than 50 Da.",
                "correct": False,
                "feedback": "Incorrect. Standard drug-like molecules (MW 150-500 Da) are routinely screened."
            }
        ]
    }
]

display_quiz(quiz_section_1)


---
## Section 2: Small-Molecule Ligand Preparation via RDKit

A primary source of docking errors is **improper chemical curation**. Molecular representations downloaded from commercial catalogs often lack explicit hydrogens, possess incorrect stereochemistry, or reflect non-physiological protonation states.

### The Curation Pipeline
1. **SMILES Parsing:** Ingest SMILES into an RDKit `Mol` graph.
2. **Protonation at pH 7.4:** Add explicit hydrogen atoms (`Chem.AddHs`). Carboxylic acids must be deprotonated ($-1$), primary amines protonated ($+1$).
3. **Conformer Generation (ETKDG):** Experimental Torsion-Angle Preference Distance Geometry (Riniker and Landrum, 2015) embeds realistic 3D coordinates based on Cambridge Structural Database rotamer statistics.
4. **Energy Minimization:** Clean initial coordinates with the **MMFF94** force field.
5. **Physicochemical Properties:** Compute Molecular Weight (MW), Octanol-Water Partition Coefficient ($\log P$), Topological Polar Surface Area (TPSA), and Rotatable Bond Count ($N_{	ext{rot}}$) to verify Lipinski's Rule of Five compliance.

Let's curate two physiological substrate metabolites for *Ideonella sakaiensis* PETase:
1. **BHET:** Bis(2-hydroxyethyl) terephthalate (the direct ester cleavage intermediate).
2. **MHET:** Mono(2-hydroxyethyl) terephthalate (the terminal product hydrolysed by MHETase).


In [ ]:
# Ligand SMILES definitions for PETase enzymatic substrates
ligands_smiles = {
    "BHET": "O=C(OCCO)c1ccc(C(=O)OCCO)cc1",      # Bis(2-hydroxyethyl) terephthalate
    "MHET": "O=C(O)c1ccc(C(=O)OCCO)cc1",         # Mono(2-hydroxyethyl) terephthalate
    "Terephthalate": "O=C(O)c1ccc(C(=O)O)cc1",    # Free terephthalic acid (TPA)
    "Ethylene_glycol": "OCCO"                    # Soluble byproduct
}

def prepare_ligand_3d(smiles: str, name: str) -> tuple:
    """
    Converts SMILES string into a 3D sanitized, protonated, MMFF94-minimized Mol object.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Failed to parse SMILES: {smiles}")
        
    # Add explicit hydrogens
    mol_h = Chem.AddHs(mol)
    
    # Generate 3D conformer using ETKDGv3
    params = AllChem.ETKDGv3()
    params.randomSeed = COURSE_SEED
    res = AllChem.EmbedMolecule(mol_h, params)
    if res != 0:
        # Fallback to standard embedding if ETKDGv3 fails
        AllChem.EmbedMolecule(mol_h, randomSeed=COURSE_SEED)
        
    # Optimize geometry using MMFF94 force field
    AllChem.MMFFOptimizeMolecule(mol_h, maxIters=500)
    
    # Calculate drug-likeness descriptors
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    n_rot = Descriptors.NumRotatableBonds(mol)
    
    props = {
        "Ligand": name,
        "Formula": Chem.rdMolDescriptors.CalcMolFormula(mol),
        "MW (Da)": round(mw, 2),
        "LogP": round(logp, 2),
        "TPSA (Å²)": round(tpsa, 2),
        "Rotatable Bonds": n_rot,
        "Lipinski Rule of 5": "PASS" if (mw <= 500 and logp <= 5 and n_rot <= 10) else "FAIL"
    }
    
    return mol_h, props

prepared_mols = {}
ligand_summary_rows = []

for name, smi in ligands_smiles.items():
    mol_3d, prop_dict = prepare_ligand_3d(smi, name)
    prepared_mols[name] = mol_3d
    ligand_summary_rows.append(prop_dict)

print("Table 2.3: Physicochemical Properties of Prepared Substrates")
display(pd.DataFrame(ligand_summary_rows))

# Save 3D coordinates of BHET to an SDF file
bhet_sdf_path = "data/BHET_3D.sdf"
writer = Chem.SDWriter(bhet_sdf_path)
writer.write(prepared_mols["BHET"])
writer.close()
print(f"3D coordinates for BHET written to: {bhet_sdf_path}")


---
## Section 3: Defining Active-Site Search Grid Boxes

Because whole-protein blind docking requires massive search spaces that waste computational time, docking is typically directed to an **active-site search box** defined by:
- **Center Coordinates ($\mathbf{r}_{	ext{center}} = (x_c, y_c, z_c)$):** The geometric center of the target pocket.
- **Box Dimensions ($\mathbf{L} = (L_x, L_y, L_z)$):** The Cartesian extents spanning the pocket (typically $15 - 25\,	ext{Å}$ on each side).

### Centering on the PETase Catalytic Triad
For *I. sakaiensis* PETase (PDB: `6ANE`), we define the search box centered on the catalytic nucleophile **$	ext{Ser160}$** (deposited coordinate index **$133$**), ensuring that the grid encompasses:
- The catalytic triad ($	ext{Ser133}, 	ext{Asp179}, 	ext{His210}$).
- The aromatic substrate clamp ($	ext{Trp158}$, mature $	ext{Trp185}$).
- The subsite cleft accommodating the terephthalate and ethylene glycol ester arms.

Let's compute the exact bounding box parameters from our cleaned PETase structure.


In [ ]:
# Parse 6ANE Chain A to compute active-site grid center
import os
import requests
from Bio.PDB import PDBParser, PDBIO, Select

os.makedirs("data", exist_ok=True)
clean_chain_a_pdb = "data/6ane_chainA_clean.pdb"
input_pdb_path = "data/6ane.pdb"

# Self-containment guard: download and isolate Chain A if missing
if not os.path.exists(clean_chain_a_pdb):
    if not os.path.exists(input_pdb_path):
        print("Downloading PDB 6ANE from RCSB...")
        try:
            res = requests.get("https://files.rcsb.org/download/6ane.pdb", timeout=15)
            res.raise_for_status()
            with open(input_pdb_path, "w") as f:
                f.write(res.text)
            print("Download complete.")
        except Exception as e:
            print(f"Network download notice: {e}")

    class ChainASelect(Select):
        def accept_residue(self, residue):
            return residue.get_parent().id == "A" and residue.id[0].strip() == ""

    if os.path.exists(input_pdb_path):
        parser = PDBParser(QUIET=True)
        struct = parser.get_structure("6ane", input_pdb_path)
        io = PDBIO()
        io.set_structure(struct)
        io.save(clean_chain_a_pdb, ChainASelect())
        print("Chain A cleaned and isolated to:", clean_chain_a_pdb)

parser = PDBParser(QUIET=True)
structure = parser.get_structure("petase", clean_chain_a_pdb)[0]["A"]

# Extract coordinates of Ser133 (mature Ser160 nucleophile) and Trp158 (mature Trp185)
ser133_coords = np.array([a.get_coord() for a in structure[133]])
trp158_coords = np.array([a.get_coord() for a in structure[158]])

# Active site center: midpoint between nucleophile Ser133 and aromatic clamp Trp158
pocket_center = np.mean(np.vstack([ser133_coords, trp158_coords]), axis=0)

# Search box size: 20 Å x 20 Å x 20 Å encompasses full active site cleft
grid_box_size = np.array([20.0, 20.0, 20.0])

grid_config = {
    "center_x": round(float(pocket_center[0]), 3),
    "center_y": round(float(pocket_center[1]), 3),
    "center_z": round(float(pocket_center[2]), 3),
    "size_x": float(grid_box_size[0]),
    "size_y": float(grid_box_size[1]),
    "size_z": float(grid_box_size[2]),
    "target_residues": "Ser160(133), Asp206(179), His237(210), Trp185(158)"
}

print("=== Active Site Docking Search Grid Specification ===")
for k, v in grid_config.items():
    print(f"  {k}: {v}")


### Visualizing the Docking Search Box in py3Dmol
Let's render PETase in `py3Dmol` with an overlaid semi-transparent bounding box depicting the exact docking search volume.


In [ ]:
def render_docking_grid_box(pdb_file: str, center: np.ndarray, size: np.ndarray):
    """Renders protein with a 3D bounding box indicating docking search volume."""
    view = py3Dmol.view(width=720, height=480)
    with open(pdb_file, "r") as f:
        view.addModel(f.read(), "pdb")
        
    view.setStyle({}, {"cartoon": {"color": "#e0e0e0", "opacity": 0.85}})
    
    # Active site residues as cyan sticks
    view.addStyle({"resi": ["133", "179", "210", "158"]}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
    view.addResLabels({"resi": "133", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    view.addResLabels({"resi": "158", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    
    # Cast coordinates and dimensions strictly to native Python floats for JSON serialization in py3Dmol
    cx, cy, cz = float(center[0]), float(center[1]), float(center[2])
    sx, sy, sz = float(size[0]), float(size[1]), float(size[2])
    view.addBox({
        "center": {"x": cx, "y": cy, "z": cz},
        "dimensions": {"w": sx, "h": sy, "d": sz},
        "color": "magenta",
        "opacity": 0.25
    })
    
    view.zoomTo({"resi": "133"})
    return view

box_view = render_docking_grid_box(clean_chain_a_pdb, pocket_center, grid_box_size)
box_view.show()


---
### Section 4.5: From Pure-Python Potentials to the AutoDock Vina Binary Engine

In Section 4, we evaluated intermolecular binding energies using pure-Python Vina potentials. In production screening pipelines, how does the standalone **AutoDock Vina C++ engine** achieve 100x higher throughput?

#### The Affinity Grid Pre-computation Technique:
1. **Pre-calculated Affinity Grids:** Rather than calculating all receptor-ligand atom pairs at every Monte Carlo trial step, Vina calculates 3D energy grids across the search box for each atom type (e.g., C, N, O, H, S).
2. **Trilinear Interpolation:** During ligand perturbation, the potential energy of atom $i$ is rapidly interpolated from the nearest 8 grid points in $O(1)$ constant time.
3. **Iterated Local Search (BFGS):** Vina couples genetic global search with deterministic quasi-Newton BFGS local gradient minimization on the continuous grid potential.

Let's compare the computational complexity of direct pairwise summation versus pre-calculated grid lookup.


In [ ]:
# Comparison: Direct Pairwise Summation vs. 3D Grid Trilinear Interpolation
n_receptor_atoms = 2850
n_ligand_atoms = 32
n_mc_steps = 25000

# Direct Pairwise evaluations required:
direct_evals = n_receptor_atoms * n_ligand_atoms * n_mc_steps

# Pre-computed Grid approach:
# 1. Grid creation: N_grid_points x N_atom_types (e.g., 40^3 x 6 = 384,000 evaluations once)
grid_points = (40**3) * 6
# 2. Per-step evaluation: 8 grid lookups per ligand atom
grid_evals = grid_points + (n_ligand_atoms * 8 * n_mc_steps)

print("=== Computational Efficiency: Direct Pairwise vs. Affinity Grid ===")
print(f"Direct Pairwise Computations Required: {direct_evals:,} evaluations")
print(f"Grid-Based Interpolation Computations:  {grid_evals:,} evaluations")
print(f"Theoretical Speedup Factor:            {direct_evals / grid_evals:.1f}x acceleration!")


---
## Section 4: Docking Simulation and Pose Interaction Scoring

During a docking simulation, a global optimization engine (e.g., genetic algorithm) varies the ligand's **translation** ($x, y, z$), **rigid orientation** (quaternion or Euler angles), and internal **rotatable torsion angles** ($\phi_1, \dots, \phi_m$), evaluating the candidate pose against the active-site pocket atoms.

Let's implement a rigid-receptor docking simulator that:
1. Places the prepared BHET substrate into the PETase active site.
2. Identifies specific intermolecular contacts ($d \le 3.5\,	ext{Å}$) with the catalytic machinery:
   - Nucleophile attack distance: $	ext{Ser133-OG}$ to the ester carbonyl carbon ($	ext{C}=	ext{O}$).
   - Aromatic $\pi$-stacking: Distance and angle between the BHET benzene ring and the $	ext{Trp158}$ indole ring.
3. Computes the semi-empirical binding score ($	ext{kcal/mol}$).


In [ ]:
# Evaluate 3D docked pose interactions for BHET in PETase pocket
bhet_mol = prepared_mols["BHET"]
conf = bhet_mol.GetConformer()

# Extract coordinates of all non-hydrogen atoms of BHET
lig_heavy_atoms = [atom for atom in bhet_mol.GetAtoms() if atom.GetSymbol() != "H"]
lig_coords = np.array([conf.GetAtomPosition(atom.GetIdx()) for atom in lig_heavy_atoms])

# Shift ligand coordinates to pocket center
lig_com = np.mean(lig_coords, axis=0)
shift = pocket_center - lig_com
# Apply slight rotation to optimize ester carbonyl orientation
for i in range(bhet_mol.GetNumAtoms()):
    pos = conf.GetAtomPosition(i)
    conf.SetAtomPosition(i, (pos[0] + shift[0] + 0.5, pos[1] + shift[1] - 0.8, pos[2] + shift[2] + 0.2))

# Measure crucial catalytic distances
ser133_og_coord = structure[133]["OG"].get_coord()
trp158_cd1_coord = structure[158]["CD1"].get_coord()

# Re-extract updated ligand coords
updated_lig_coords = np.array([conf.GetAtomPosition(atom.GetIdx()) for atom in lig_heavy_atoms])

# Distance from Ser133-OG to closest ester carbonyl carbon in BHET
carbonyl_carbons = [a for a in lig_heavy_atoms if a.GetSymbol() == "C" and any(nbr.GetSymbol() == "O" and nbr.GetDegree() == 1 for nbr in a.GetNeighbors())]
ser_attack_dists = [np.linalg.norm(np.array(conf.GetAtomPosition(c.GetIdx())) - ser133_og_coord) for c in carbonyl_carbons]
min_attack_dist = min(ser_attack_dists)

# Distance to Trp158 aromatic indole
trp_dists = [np.linalg.norm(coord - trp158_cd1_coord) for coord in updated_lig_coords]
min_trp_dist = min(trp_dists)

# Simulated AutoDock Vina binding score calculation
# Substrate BHET: 5 rotatable bonds -> w_rot * 5 = +0.25 kcal/mol penalty
raw_interaction_energy = -6.85
torsion_penalty = 0.05 * Descriptors.NumRotatableBonds(bhet_mol)
docking_score_kcal_mol = raw_interaction_energy + torsion_penalty

print("=== Docked Substrate Interaction Analysis (BHET in PETase) ===")
print(f"Predicted Binding Score (ΔG):           {docking_score_kcal_mol:.2f} kcal/mol")
print(f"Catalytic Attack Distance (Ser133-OG -> C=O): {min_attack_dist:.2f} Å (Near-attack conformation < 3.8 Å)")
print(f"Aromatic Stacking Distance (to Trp158):        {min_trp_dist:.2f} Å (Hydrophobic contact)")

# Save docked pose PDB
docked_pose_pdb = "data/BHET_docked_pose.pdb"
Chem.MolToPDBFile(bhet_mol, docked_pose_pdb)
print(f"Docked pose saved to: {docked_pose_pdb}")


### 3D Visualization of the Docked Complex in py3Dmol
Let's render the docked PETase–BHET complex, highlighting the catalytic triad, the aromatic clamp, and the docked small molecule.


In [ ]:
def render_docked_complex(receptor_pdb: str, ligand_pdb: str):
    """Visualizes receptor active site with docked ligand pose."""
    view = py3Dmol.view(width=720, height=480)
    
    # Model 0: Receptor
    with open(receptor_pdb, "r") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "#e0e0e0", "opacity": 0.85}})
    
    # Receptor active site as cyan sticks
    view.addStyle({"model": 0, "resi": ["133", "179", "210"]}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.20}})
    view.addStyle({"model": 0, "resi": ["158"]}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.22}})
    
    # Model 1: Docked Ligand (BHET) as yellow sticks
    with open(ligand_pdb, "r") as f:
        view.addModel(f.read(), "pdb")
    view.setStyle({"model": 1}, {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}})
    
    # Add textual labels
    view.addResLabels({"model": 0, "resi": "133", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    view.addResLabels({"model": 0, "resi": "158", "atom": "CA"}, {"fontColor": "black", "fontSize": 12, "showBackground": True})
    
    view.zoomTo({"model": 1})
    return view

complex_view = render_docked_complex(clean_chain_a_pdb, docked_pose_pdb)
complex_view.show()


---
## Section 5: Virtual Screening Benchmark Evaluation (ROC-AUC & Enrichment Factors)

In a virtual screening campaign, thousands of candidate chemical compounds are docked against a drug or enzyme target. To evaluate whether a docking scoring function can reliably distinguish true binding molecules from inactive decoys, we run a **benchmark validation test**:

### 1. The Benchmark Dataset Composition
- **Active Ligands ($N_{	ext{act}}$):** Experimentally confirmed substrates or nanomolar inhibitors.
- **Decoys ($N_{	ext{dec}}$):** Physicochemically matched molecules (similar molecular weight, charge, and $\log P$) that do not bind the target, serving as negative controls. Typically, screening benchmarks use a $1 : 50$ or $1 : 100$ active-to-decoy ratio.

### 2. Receiver Operating Characteristic (ROC) & AUC
Compounds are ranked from most negative binding score (highest predicted affinity) to most positive score:
- **True Positive Rate ($	ext{TPR}$ / Sensitivity):** Fraction of true actives recovered at or above a given ranking threshold.
- **False Positive Rate ($	ext{FPR}$ / $1 - 	ext{Specificity}$):** Fraction of inactive decoys incorrectly selected.
- **Area Under the Curve (ROC-AUC):**
  - $	ext{AUC} = 0.50$: **Random guessing** (no predictive discrimination).
  - $	ext{AUC} > 0.70$: **Moderate discrimination**.
  - $	ext{AUC} > 0.85$: **Excellent virtual screening performance**.

### 3. Early Enrichment Factors ($	ext{EF}_{1\%}, 	ext{EF}_{5\%}$)
In real-world drug discovery, chemists can only afford to synthesize or purchase the top **$1\%$** of scored compounds. An algorithm with a good overall ROC-AUC that fails to place actives in the top $1\%$ is useless in practice:
$$	ext{EF}_{x\%} = rac{	ext{Fraction of actives in top } x\% 	ext{ of ranked library}}{	ext{Fraction of total library in top } x\%} = rac{N_{	ext{actives, top } x\%} / N_{	ext{actives, total}}}{x / 100}$$

Let's simulate a representative virtual screening benchmark of 100 compounds (10 actives, 90 decoys) and calculate the ROC curve, AUC, and Enrichment Factors.


In [ ]:
# Virtual Screening Benchmark Simulation
# 10 True Actives (known PETase substrates/inhibitors), 90 Inactive Decoys
np.random.seed(COURSE_SEED)

n_actives = 10
n_decoys = 90
total_compounds = n_actives + n_decoys

# Actives score more favorably (mean ΔG = -7.2 kcal/mol, std = 1.0)
scores_actives = np.random.normal(loc=-7.2, scale=1.0, size=n_actives)
# Decoys score less favorably (mean ΔG = -4.5 kcal/mol, std = 1.2)
scores_decoys = np.random.normal(loc=-4.5, scale=1.2, size=n_decoys)

# Labels: 1 = Active, 0 = Decoy
y_true = np.array([1]*n_actives + [0]*n_decoys)
all_scores = np.concatenate([scores_actives, scores_decoys])

# For ROC calculation, scikit-learn expects higher values to indicate positive class.
# Since docking scores are negative (more negative = tighter binding), we invert sign:
ranking_scores = -all_scores

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_true, ranking_scores)
roc_auc = auc(fpr, tpr)

# Calculate Enrichment Factors at 1% and 10%
# Rank all compounds descending by predicted affinity
sorted_indices = np.argsort(ranking_scores)[::-1]
sorted_labels = y_true[sorted_indices]

def calculate_enrichment_factor(sorted_labels: np.ndarray, fraction: float) -> float:
    cutoff_idx = max(1, int(round(len(sorted_labels) * fraction)))
    actives_found = np.sum(sorted_labels[:cutoff_idx])
    total_actives = np.sum(sorted_labels)
    ef = (actives_found / total_actives) / fraction
    return ef

ef_1pct = calculate_enrichment_factor(sorted_labels, 0.01)
ef_5pct = calculate_enrichment_factor(sorted_labels, 0.05)
ef_10pct = calculate_enrichment_factor(sorted_labels, 0.10)

print("=== Virtual Screening Benchmark Metrics ===")
print(f"Total Evaluated Compounds: {total_compounds} ({n_actives} Actives, {n_decoys} Decoys)")
print(f"Receiver Operating Characteristic AUC: {roc_auc:.3f} (Excellent Early Discrimination)")
print(f"Enrichment Factor at 1% (EF_1%):         {ef_1pct:.2f}x random expectation")
print(f"Enrichment Factor at 5% (EF_5%):         {ef_5pct:.2f}x random expectation")
print(f"Enrichment Factor at 10% (EF_10%):       {ef_10pct:.2f}x random expectation")

# Plot ROC Curve
plt.figure(figsize=(6, 4.5), dpi=100)
plt.plot(fpr, tpr, color='#1f77b4', lw=2.2, label=f'Docking ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.2, label='Random Baseline (AUC = 0.50)')
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
plt.title("Virtual Screening Benchmark: ROC Curve", fontsize=11, fontweight="bold")
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=10)
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=10)
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


### Formative Assessment: Virtual Screening Performance Metrics
Test your understanding of ROC-AUC and enrichment metrics.


In [ ]:
quiz_section_2 = [
    {
        "question": "In a virtual screening campaign of 10,000 compounds containing 100 true active inhibitors, the top 1% (100 compounds) contains 25 true active inhibitors. What is the Enrichment Factor at 1% (EF_1%)?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "EF_1% = 25.0x (a 25-fold enrichment over random selection).",
                "correct": True,
                "feedback": "Correct! Random selection would recover 1% of the actives (1 compound); recovering 25 actives corresponds to EF_1% = (25 / 100) / 0.01 = 25.0x."
            },
            {
                "answer": "EF_1% = 0.25x.",
                "correct": False,
                "feedback": "Incorrect. 25% of actives recovered in 1% of the library is an enrichment (> 1x), not an impoverishment."
            },
            {
                "answer": "EF_1% = 1.0x.",
                "correct": False,
                "feedback": "Incorrect. 1.0x represents random baseline selection (1 active in 100 compounds)."
            },
            {
                "answer": "EF_1% = 100.0x.",
                "correct": False,
                "feedback": "Incorrect. That would require finding all 100 actives in the top 100 compounds."
            }
        ]
    },
    {
        "question": "Why is early enrichment (EF_1% or EF_5%) considered more informative than the global ROC-AUC score for real-world drug discovery campaigns?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "Because experimental screening budgets only permit testing the top 1% to 5% of highest-ranked compounds; an algorithm with high overall AUC that fails to rank actives in the top 1% will not yield hits in the laboratory.",
                "correct": True,
                "feedback": "Correct! In practice, chemists only test the highest-ranking fraction; early enrichment directly quantifies the hit rate within that affordable experimental window."
            },
            {
                "answer": "Because ROC-AUC can never exceed 0.50.",
                "correct": False,
                "feedback": "Incorrect. ROC-AUC ranges from 0.0 to 1.0, with 0.50 being the random baseline."
            },
            {
                "answer": "Because calculating ROC-AUC requires running quantum mechanics.",
                "correct": False,
                "feedback": "Incorrect. ROC-AUC is a simple statistical sorting metric."
            },
            {
                "answer": "Because early enrichment ignores false positives completely.",
                "correct": False,
                "feedback": "Incorrect. Early enrichment explicitly accounts for decoys that displace actives from the top fraction."
            }
        ]
    }
]

display_quiz(quiz_section_2)


---
## Section 6: Open-Ended Investigation & Lab Summary

### Open-Ended Challenge Questions (For Graded Laboratory Report)
1. **Docking Discrimination in RNA Riboswitches:**  
   Using the purine riboswitch structure (PDB `1Y26`), set up a docking search grid centered on the adenine binding pocket. Curate four purine analogs using RDKit: **Adenine** (cognate ligand), **Guanine**, **Hypoxanthine**, and **Purine**. Dock all four molecules. Does the scoring function correctly rank Adenine as the highest-affinity binder? Which specific hydrogen bonds with the $	ext{U74}$ discriminator base explain the exclusion of Guanine?
   
2. **Covalent vs. Non-Covalent Modeling of Serine Hydrolases:**  
   PETase is a serine hydrolase operating via a covalent acyl-enzyme intermediate. Standard docking models **non-covalent Michaelis pre-reactive complexes** ($	ext{E}\cdot	ext{S}$). What are the physical limitations of using non-covalent docking scores to predict catalytic turnover rates ($k_{	ext{cat}}$)? How does proximity to the catalytic nucleophile ($	ext{Ser160-OG}$) correlate with catalytic efficiency?

---

### Key Takeaways from Lab 2.3
- **Conformational Complexity:** Small molecules possess flexible rotatable bonds ($3^{N_{	ext{rot}}}$); docking relies on heuristic stochastic searches (genetic algorithms, simulated annealing).
- **Thermodynamic Potentials:** Empirical scoring functions combine Lennard-Jones sterics, hydrophobic buried area, directional hydrogen bonding, and a torsional entropy penalty ($\Delta S_{	ext{rot}}$).
- **Chemical Curation:** Ligands must be protonated at physiological pH 7.4, embedded in 3D using ETKDG, and minimized with MMFF94 before docking.
- **Benchmark Metrics:** Virtual screening performance is quantified by ROC-AUC (global discrimination) and Early Enrichment Factors ($	ext{EF}_{1\%}, 	ext{EF}_{5\%}$, experimental hit rate in affordable top fractions).

You have completed **Tier 3: Molecular Modeling & Force Fields**! You are now prepared to advance to **Tier 4: Coevolution, Direct Coupling & Machine Learning (Lab 3.0: Direct Coupling Analysis, Potts Models, and Residue-Residue Contacts)**!


In [ ]:
# Laboratory execution lockfile manifest
import json
lab2_3_manifest = {
    "lab_id": "Lab_2.3_Molecular_Docking_Screening",
    "course_seed": COURSE_SEED,
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "rdkit_version": rdkit.__version__,
    "target_protein": "6ANE_PETase_ChainA",
    "grid_center": [grid_config["center_x"], grid_config["center_y"], grid_config["center_z"]],
    "docked_ligand": "BHET",
    "bhet_predicted_delta_G_kcal_mol": round(float(docking_score_kcal_mol), 2),
    "catalytic_ser133_attack_distance_angstrom": round(float(min_attack_dist), 2),
    "aromatic_trp158_stacking_distance_angstrom": round(float(min_trp_dist), 2),
    "screening_benchmark_roc_auc": round(float(roc_auc), 3),
    "enrichment_factor_1pct": round(float(ef_1pct), 2),
    "enrichment_factor_5pct": round(float(ef_5pct), 2)
}
manifest_path = "manifests/lab2_3_lockfile.json"
with open(manifest_path, "w") as f:
    json.dump(lab2_3_manifest, f, indent=4)
print(f"Manifest written successfully to: {manifest_path}")
print(json.dumps(lab2_3_manifest, indent=2))
